# 02B — Shuttlecock Tracking (ShuttleSet)

Runs **TrackNetV4** on the 11 target ShuttleSet matches to extract per-frame shuttle positions.

**Model**: `tracknet-series-pytorch` (same as FineBadminton notebook)
- Input: 3 consecutive RGB frames stacked → `[1, 9, 288, 512]`
- Output: 3 heatmaps with motion attention → argmax of center-frame heatmap gives shuttle pixel (x, y)

**Rally structure**: Derived from per-match set CSVs (`datasets/ShuttleSet/set/{match_id}/set{N}.csv`),
which contain `rally` and `frame_num` columns. We group shots by rally and find the frame range.

**Output per rally**: `datasets_preprocessing/shuttleset_shuttles/{match_id}_s{set}r{rally}.npy`
- Shape: `(T, 3)` — columns are `[x, y, visible]`
- `x, y` in original pixel space (1920 × 1080, ShuttleSet coordinate system)
- `visible = 1.0` if max heatmap confidence ≥ threshold, else `0.0`
- `T` = number of available frames in the rally

**Steps**:
1. Download pretrained weights (once)
2. Load V4 model & verify forward pass
3. Build rally frame grouping from set CSVs
4. Extract shuttle trajectories for target matches
5. Verify quality — overlay on frames, summary statistics

In [ ]:
import os, sys, zipfile
from pathlib import Path

# ── Colab detection ───────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    ZIP_PATH     = DRIVE_ROOT / 'baddiev2_colab.zip'

    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print(f'Extracted to {PROJECT_PATH}')
    else:
        print('Project already extracted.')

    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    # ── Paths on Drive ────────────────────────────────────────────────────
    SS_FRAMES   = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_frames'
    SS_SHUTTLES = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_shuttles'
    SS_CSV_ROOT = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    WEIGHTS_V4  = DRIVE_ROOT / 'tracknet-v4_best-model.pth'

    # TrackNet model code
    TRACKNET_DIR = PROJECT_PATH / 'datasets' / 'tracknet-series-pytorch'
    sys.path.insert(0, str(TRACKNET_DIR))

    print(f'Drive root  : {DRIVE_ROOT}')
    print(f'SS Frames   : {SS_FRAMES}')
    print(f'SS Shuttles : {SS_SHUTTLES}')
    print(f'SS CSV root : {SS_CSV_ROOT}')
    print(f'CSV exists  : {SS_CSV_ROOT.exists()}')
else:
    sys.path.insert(0, '..')
    sys.path.insert(0, '../datasets/tracknet-series-pytorch')
    from src.config import SS_FRAMES, SS_SHUTTLES, SS_CSV_ROOT, PROJECT_ROOT
    WEIGHTS_V4 = PROJECT_ROOT / 'datasets' / 'tracknet-series-pytorch' / 'checkpoints' / 'tracknet-v4_best-model.pth'
    WEIGHTS_V4.parent.mkdir(parents=True, exist_ok=True)
    print('Local run — using paths from config.py')

import csv
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm

SS_SHUTTLES.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {DEVICE}')
print(f'Shuttle output dir: {SS_SHUTTLES}')
print(f'SS Frames dir: {SS_FRAMES}')
print(f'SS CSV root: {SS_CSV_ROOT}')

## 1. Download Pretrained Weights

Downloads `tracknet-v4_best-model.pth` from the GitHub release (once — skips if already present).

In [ ]:
import urllib.request

RELEASE = 'https://github.com/AnInsomniacy/tracknet-series-pytorch/releases/download/v1.0.1'

if WEIGHTS_V4.exists():
    print(f'  [SKIP] Already downloaded ({WEIGHTS_V4.stat().st_size/1e6:.1f} MB)')
else:
    url = f'{RELEASE}/tracknet-v4_best-model.pth'
    print(f'  Downloading tracknet-v4_best-model.pth...')
    urllib.request.urlretrieve(url, WEIGHTS_V4)
    print(f'  Saved → {WEIGHTS_V4} ({WEIGHTS_V4.stat().st_size/1e6:.1f} MB)')

## 2. Load TrackNetV4 & Verify Forward Pass

In [ ]:
from model.tracknet_v4 import TrackNet

model = TrackNet().to(DEVICE)
checkpoint = torch.load(WEIGHTS_V4, map_location=DEVICE)
state_dict = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(state_dict)
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f'TrackNetV4 loaded — {params:,} parameters')

# Sanity-check forward pass
dummy = torch.zeros(1, 9, 288, 512).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Forward pass: {tuple(dummy.shape)} → {tuple(out.shape)}')
print('Output range:', out.min().item(), '–', out.max().item())

## 3. Target Matches & Rally Grouping from CSVs

The 21 target matches (19 train + 2 held-out). Rally boundaries come from
the per-match set CSVs in `datasets/ShuttleSet/set/{match_id}/set{N}.csv`.

In [ ]:
# ── Target matches (19 train + 2 held-out) ───────────────────────────────────
TARGET_MATCHES = [
    # Train — original 8 (GDINO skeletons done)
    "Anders_Antonsen_Viktor_Axelsen_HSBC_BWF_WORLD_TOUR_FINALS_2020_Finals",
    "Anthony_Sinisuka_GINTING_Anders_ANTONSEN_Indonesia_Masters_2020_Final",
    "Anthony_Sinisuka_GINTING_Viktor_AXELSEN _Indonesia_Masters_2020_SemiFinals",
    "CHOU_Tien_Chen_Anders_ANTONSEN_Fuzhou_Open_2019_Semi-finals",
    "Ng_Ka_Long_Angus_Lee_Cheuk_Yiu_YONEX_Thailand_Open_2021_QuarterFinals",
    "Viktor_AXELSEN _SHI_Yu_Qi_All_England_Open_2020_QuarterFinals",
    "Viktor_AXELSEN_CHEN_Long_Malaysia_Masters_2020_QuarterFinals",
    "Viktor_AXELSEN_NG_Ka_Long_Angus_Malaysia_Masters_2020_SemiFinals",
    # Train — 11 new men's matches (skeletons pending)
    "Anders_Antonsen_Sameer_Verma_TOYOTA_THAILAND_OPEN_2021_QuarterFinals",
    "CHOU_Tien_Chen_Jonatan_CHRISTIE_Indonesia_Open_2019_Quarter-finals",
    "Hans-Kristian_Solberg_Vittinghus_Anders_Antonsen_TOYOTA_THAILAND_OPEN_2021_SemiFinals",
    "Hans-Kristian_Solberg_Vittinghus_Lee_Cheuk_Yu_TOYOTA_THAILAND_OPEN_2021_QuarterFinals",
    "NG_Ka_Long_Angus_Jonatan_CHRISTIE_Malaysia_Masters_2020_QuarterFinals",
    "Ng_Ka_Long_Angus_Kidambi_Srikanth_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals",
    "Viktor_Axelsen_Anthony_Sinisuka_Ginting_YONEX_Thailand_Open_2021_SemiFinals",
    "Viktor_Axelsen_Hans-Kristian_Solberg_VIittinghus_TOYOTA_THAILAND_OPEN_2021_Finals",
    "Viktor_Axelsen_Jonatan_Christie_YONEX_Thailand_Open_2021_QuarterFinals",
    "Viktor_Axelsen_Liew_Daren_TOYOTA_THAILAND_OPEN_2021_QuarterFinals",
    "Viktor_Axelsen_Ng_Ka_Long_Angus_YONEX_Thailand_Open_2021_Finals",
    # Held-out (2 matches, 1,675 shots) — final evaluation
    "Anthony_Sinisuka_Ginting_Lee_Zii_Jia_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals",
    "CHEN_Long_CHOU_Tien_Chen_World_Tour_Finals_Group_Stage",
]


def build_rally_index(match_id):
    """
    Parse set CSVs to get rally frame ranges, then find actual frames on disk.

    Returns: list of dicts with keys:
        - rally_id: str like "s1r3"
        - frame_start, frame_end: int (min/max frame_num in rally)
        - frame_paths: list of (frame_num, Path) sorted by frame_num
    """
    csv_dir = SS_CSV_ROOT / match_id
    if not csv_dir.exists():
        print(f'  [WARN] No CSV dir for {match_id}')
        return []

    # Read all set CSVs → group by (set, rally) → frame range
    rallies = {}
    for csv_file in sorted(csv_dir.glob('set*.csv')):
        set_num = int(csv_file.stem.replace('set', ''))
        with open(csv_file, newline='') as f:
            for row in csv.DictReader(f):
                rally_int = int(row['rally'])
                frame_num = row.get('frame_num', '')
                if not frame_num or frame_num == '':
                    continue
                frame_num = int(float(frame_num))
                key = f's{set_num}r{rally_int}'
                if key not in rallies:
                    rallies[key] = {
                        'rally_id': key,
                        'frame_start': frame_num,
                        'frame_end': frame_num,
                    }
                rallies[key]['frame_start'] = min(rallies[key]['frame_start'], frame_num)
                rallies[key]['frame_end']   = max(rallies[key]['frame_end'],   frame_num)

    # Find actual frames on disk for each rally
    frame_dir = SS_FRAMES / match_id
    if not frame_dir.exists():
        print(f'  [WARN] No frames dir for {match_id}')
        return []

    all_frames = {}
    for fp in frame_dir.glob('frame_*.jpg'):
        fnum = int(fp.stem.split('_')[1])
        all_frames[fnum] = fp

    result = []
    for rally_id, info in sorted(rallies.items()):
        frame_paths = [
            (fnum, all_frames[fnum])
            for fnum in sorted(all_frames.keys())
            if info['frame_start'] <= fnum <= info['frame_end']
        ]
        if len(frame_paths) >= 3:  # need at least 3 for TrackNet
            info['frame_paths'] = frame_paths
            result.append(info)

    return result


# ── Audit: which targets are ready ───────────────────────────────────────────
print(f'Target matches: {len(TARGET_MATCHES)}\n')
print(f'{"Match":<70} {"CSVs":>5} {"Frames":>7} {"Shuttles":>9}')
print('-' * 95)
match_ids = []
for mid in TARGET_MATCHES:
    has_csv    = (SS_CSV_ROOT / mid).exists()
    frame_dir  = SS_FRAMES / mid
    n_frames   = len(list(frame_dir.glob('*.jpg'))) if frame_dir.exists() else 0
    n_shuttles = len([f for f in SS_SHUTTLES.glob(f'{mid}_s*r*.npy')
                      if '_frames' not in f.stem]) if SS_SHUTTLES.exists() else 0
    status = 'ready' if has_csv and n_frames > 0 else 'MISSING'
    print(f'  {mid:<68} {"Y" if has_csv else "N":>5} {n_frames:>7} {n_shuttles:>9}  {status}')
    if has_csv and n_frames > 0:
        match_ids.append(mid)

print(f'\nQueued for processing: {len(match_ids)} matches')

# Quick test
if match_ids:
    test_rallies = build_rally_index(match_ids[0])
    print(f'\nTest — {match_ids[0][:50]}:')
    print(f'  {len(test_rallies)} rallies with ≥3 frames')
    for r in test_rallies[:3]:
        print(f'  {r["rally_id"]}: frames {r["frame_start"]}–{r["frame_end"]}, '
              f'{len(r["frame_paths"])} available')
    total = sum(len(r['frame_paths']) for r in test_rallies)
    print(f'  Total frames: {total}')

## 4. Shuttle Extraction Helper

Same sliding-window approach as FineBadminton: 3 consecutive *available* frames → TrackNet → heatmap → (x, y, visible).
Output resolution is 1920×1080 (ShuttleSet native).

In [ ]:
IMG_W, IMG_H = 1920, 1080  # ShuttleSet native resolution
MODEL_W, MODEL_H = 512, 288  # TrackNet input resolution
THRESHOLD = 0.5              # min heatmap confidence to declare detection


def preprocess_frame(bgr_frame):
    """BGR frame → (3, 288, 512) float32 tensor in [0,1]."""
    resized = cv2.resize(bgr_frame, (MODEL_W, MODEL_H))
    t = torch.from_numpy(resized.astype(np.float32) / 255.0).permute(2, 0, 1)
    return t  # (3, 288, 512)


def heatmap_to_xy(heatmap, orig_w=IMG_W, orig_h=IMG_H, threshold=THRESHOLD):
    """
    Decode a single (288, 512) heatmap → (x, y, visible) in original pixel space.
    Returns (0.0, 0.0, 0.0) when max confidence < threshold.
    """
    conf = float(heatmap.max())
    if conf < threshold:
        return 0.0, 0.0, 0.0
    idx = np.unravel_index(np.argmax(heatmap), heatmap.shape)  # (row, col)
    model_y, model_x = idx
    x = model_x * (orig_w / MODEL_W)
    y = model_y * (orig_h / MODEL_H)
    return float(x), float(y), 1.0


@torch.no_grad()
def extract_shuttle_trajectory(frame_paths, batch_size=8):
    """
    Extract shuttle (x, y, visible) for every available frame in a rally.

    Uses a sliding window of 3 consecutive available frames.
    For the middle frames (1 … T-2) we use the centre heatmap (output[:, 1]).
    For frame 0 we use the first triplet's first heatmap (output[:, 0]).
    For frame T-1 we use the last triplet's last heatmap (output[:, 2]).

    Args:
        frame_paths: list of (frame_num, Path) sorted by frame_num
        batch_size: number of triplets per forward pass

    Returns:
        trajectory: np.ndarray (T, 3) — [x, y, visible]
        frame_nums: np.ndarray (T,) — frame numbers for each position
    """
    T = len(frame_paths)
    trajectory = np.zeros((T, 3), dtype=np.float32)
    frame_nums = np.array([fn for fn, _ in frame_paths], dtype=np.int32)

    if T < 3:
        return trajectory, frame_nums

    # Pre-process all frames once
    tensors = [preprocess_frame(cv2.imread(str(fp))) for _, fp in frame_paths]

    # Build sliding-window triplets
    triplets = [
        torch.cat(tensors[i:i+3], dim=0).unsqueeze(0)  # (1, 9, H, W)
        for i in range(T - 2)
    ]

    heatmaps = np.zeros((T - 2, 3, MODEL_H, MODEL_W), dtype=np.float32)

    for start in range(0, len(triplets), batch_size):
        batch = torch.cat(triplets[start:start+batch_size], dim=0).to(DEVICE)
        out   = model(batch).cpu().numpy()
        heatmaps[start:start+len(batch)] = out

    # Frame 0: use triplet-0 output channel 0
    trajectory[0] = heatmap_to_xy(heatmaps[0, 0])

    # Frames 1 … T-2: centre heatmap of each triplet
    for i in range(1, T - 1):
        trajectory[i] = heatmap_to_xy(heatmaps[i - 1, 1])

    # Frame T-1: use last triplet output channel 2
    trajectory[T - 1] = heatmap_to_xy(heatmaps[-1, 2])

    return trajectory, frame_nums


print('Helper functions defined.')
print(f'Threshold: {THRESHOLD}  |  Model res: {MODEL_W}×{MODEL_H}  |  Output space: {IMG_W}×{IMG_H}')

## 5. Extract Trajectories for Target Matches

Iterates over the 11 target matches. Saves one `.npy` per rally:
`{match_id}_s{set}r{rally}.npy` with shape `(T, 3)`, plus a companion `_frames.npy`.
Already-extracted rallies are skipped automatically.

In [ ]:
# match_ids is built in Section 3 from TARGET_MATCHES (only those with CSVs + frames)
print(f'Processing {len(match_ids)} matches\n')

stats = []

for match_id in match_ids:
    rallies = build_rally_index(match_id)

    # Filter to rallies not yet processed
    todo = []
    for r in rallies:
        out_path = SS_SHUTTLES / f'{match_id}_{r["rally_id"]}.npy'
        if not out_path.exists():
            todo.append(r)

    if not todo:
        print(f'  {match_id[:60]}: all {len(rallies)} rallies already done')
        continue

    print(f'\n{match_id}: {len(todo)}/{len(rallies)} rallies to process')

    for r in tqdm(todo, desc=f'  {match_id[:40]}'):
        out_path = SS_SHUTTLES / f'{match_id}_{r["rally_id"]}.npy'
        frames_path = SS_SHUTTLES / f'{match_id}_{r["rally_id"]}_frames.npy'

        traj, frame_nums = extract_shuttle_trajectory(r['frame_paths'])
        np.save(out_path, traj)
        np.save(frames_path, frame_nums)

        visible = int(traj[:, 2].sum())
        T = len(traj)
        rate = 100 * visible / T if T > 0 else 0
        stats.append({
            'match': match_id, 'rally': r['rally_id'],
            'T': T, 'detected': visible, 'rate': rate
        })

# Summary
if stats:
    total_T = sum(s['T'] for s in stats)
    total_vis = sum(s['detected'] for s in stats)
    print(f'\nProcessed {len(stats)} rallies, {total_T} frames')
    print(f'Overall detection rate: {total_vis}/{total_T} ({100*total_vis/total_T:.1f}%)')

    for s in stats[:10]:
        print(f'  {s["match"][:50]} {s["rally"]}: T={s["T"]}, detected={s["detected"]}/{s["T"]} ({s["rate"]:.0f}%)')
    if len(stats) > 10:
        print(f'  ... and {len(stats) - 10} more rallies')
else:
    print('All rallies already extracted — nothing to do.')

## 6. Verify — Overlay on Actual Frames

Draws the detected shuttle position on 8 evenly-spaced frames from a rally,
with a 10-frame trailing arc to visualise direction.

In [ ]:
TRAIL = 10

# Find a completed rally to visualise
done_files = sorted(SS_SHUTTLES.glob('*_s*r*.npy'))
done_files = [f for f in done_files if '_frames' not in f.stem]

if not done_files:
    print('No trajectory files yet — run Section 5 first.')
else:
    for traj_file in done_files[:2]:
        rally_key = traj_file.stem  # e.g. "match_id_s1r3"
        traj = np.load(traj_file)
        frames_file = traj_file.parent / f'{rally_key}_frames.npy'
        frame_nums = np.load(frames_file)
        
        # Find match_id and rally_id from filename
        # rally_id is the last part like s1r3
        parts = rally_key.rsplit('_', 1)
        match_id, rally_id = parts[0], parts[1]
        
        frame_dir = SS_FRAMES / match_id
        T = len(traj)
        
        # 8-frame grid
        idxs = np.linspace(0, T - 1, 8, dtype=int)
        fig, axes = plt.subplots(2, 4, figsize=(22, 10))
        
        for ax, t in zip(axes.flat, idxs):
            fnum = frame_nums[t]
            fp = frame_dir / f'frame_{fnum:06d}.jpg'
            if fp.exists():
                img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, f'frame {fnum}\nnot found', ha='center', va='center')
            
            # Trail
            for dt in range(TRAIL, 0, -1):
                ti = t - dt
                if ti >= 0 and traj[ti, 2] > 0:
                    alpha = dt / TRAIL
                    ax.scatter(traj[ti, 0], traj[ti, 1],
                               s=20, color=(1, 1, 0), alpha=alpha*0.6)
            
            # Current position
            if traj[t, 2] > 0:
                ax.scatter(traj[t, 0], traj[t, 1],
                           s=80, color='yellow', edgecolors='red', linewidths=1.5, zorder=5)
                ax.set_title(f'f={fnum} ({traj[t,0]:.0f},{traj[t,1]:.0f})', fontsize=8)
            else:
                ax.set_title(f'f={fnum} [not detected]', fontsize=8, color='gray')
            
            ax.axis('off')
        
        visible = int(traj[:, 2].sum())
        fig.suptitle(f'{match_id} / {rally_id} — shuttle overlay ({visible}/{T} detected)', fontsize=11)
        plt.tight_layout()
        plt.show()

## 7. Verify — Full Rally Trajectory Plot

Shows the shuttle's x-y path across the entire rally to sanity-check trajectory shape.

In [ ]:
done_files = sorted(SS_SHUTTLES.glob('*_s*r*.npy'))
done_files = [f for f in done_files if '_frames' not in f.stem]

if not done_files:
    print('No trajectory files yet.')
else:
    for traj_file in done_files[:2]:
        rally_key = traj_file.stem
        traj = np.load(traj_file)
        frames_file = traj_file.parent / f'{rally_key}_frames.npy'
        frame_nums = np.load(frames_file)
        
        vis = traj[:, 2] > 0
        xs, ys = traj[vis, 0], traj[vis, 1]
        ts = np.where(vis)[0]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # x-y trajectory
        sc = axes[0].scatter(xs, ys, c=ts, cmap='plasma', s=8, alpha=0.7)
        axes[0].invert_yaxis()
        axes[0].set_xlim(0, IMG_W); axes[0].set_ylim(IMG_H, 0)
        axes[0].set_xlabel('x (px)'); axes[0].set_ylabel('y (px) ↓')
        axes[0].set_title(f'{rally_key} — shuttle path (colour=time)')
        axes[0].set_facecolor('#0a0f1a')
        plt.colorbar(sc, ax=axes[0], label='frame idx')
        
        # x over time
        axes[1].plot(ts, xs, lw=1, color='#60a5fa')
        axes[1].set_xlabel('Frame idx'); axes[1].set_ylabel('x (px)')
        axes[1].set_title('Lateral position over time')
        
        # y over time
        axes[2].plot(ts, ys, lw=1, color='#f472b6')
        axes[2].invert_yaxis()
        axes[2].set_xlabel('Frame idx'); axes[2].set_ylabel('y (px) ↓')
        axes[2].set_title('Depth position over time')
        
        detection_rate = vis.mean() * 100
        plt.suptitle(f'{rally_key}: {vis.sum()}/{len(vis)} frames detected ({detection_rate:.1f}%)', fontsize=10)
        plt.tight_layout()
        plt.show()

## 8. Summary — All Processed Rallies

In [ ]:
# Summary scoped to TARGET_MATCHES only
print(f'Shuttle trajectories for {len(TARGET_MATCHES)} target matches:\n')

total_vis = total_frames = 0
match_stats = defaultdict(lambda: {'rallies': 0, 'frames': 0, 'detected': 0})

for mid in TARGET_MATCHES:
    pattern = f'{mid}_s*r*.npy'
    for f in sorted(SS_SHUTTLES.glob(pattern)):
        if '_frames' in f.stem:
            continue
        t = np.load(f)
        total_frames += len(t)
        total_vis += int(t[:, 2].sum())
        match_stats[mid]['rallies'] += 1
        match_stats[mid]['frames'] += len(t)
        match_stats[mid]['detected'] += int(t[:, 2].sum())

for mid in TARGET_MATCHES:
    s = match_stats[mid]
    if s['frames'] > 0:
        rate = 100 * s['detected'] / s['frames']
        print(f'  {mid[:65]:65s}  rallies={s["rallies"]:3d}  frames={s["frames"]:5d}  detected={rate:.0f}%')
    else:
        print(f'  {mid[:65]:65s}  NOT YET EXTRACTED')

if total_frames:
    n_rallies = sum(s['rallies'] for s in match_stats.values())
    print(f'\nOverall: {n_rallies} rallies, {total_frames} frames, '
          f'detection rate {total_vis}/{total_frames} ({100*total_vis/total_frames:.1f}%)')
else:
    print('\nNo shuttle trajectories extracted yet.')